In [ ]:
from langgraph.graph import StateGraph,MessagesState,START,END
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage,AIMessage
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages.utils import trim_messages,count_tokens_approximately
from dotenv import load_dotenv
load_dotenv()

In [ ]:
model = ChatGroq( model="openai/gpt-oss-20b",
    temperature=0.7)

In [ ]:
MAX_TOKENS=2000

In [ ]:
checkpointer = InMemorySaver()

In [ ]:
def call_llm(state:MessagesState):

    
    trimmed_messages = trim_messages(state['messages'],
                                     strategy="last",
                                     token_counter=count_tokens_approximately,
                                     max_tokens=MAX_TOKENS)

    print("current Token count->:",count_tokens_approximately(messages=trimmed_messages))

    for msg in trimmed_messages:
        print(msg.content)
        
    res = model.invoke(trimmed_messages)

    return {"messages":[res]}

    

In [ ]:
graph = StateGraph(MessagesState)

graph.add_node("call_llm",call_llm)

graph.add_edge(START,'call_llm')

In [ ]:
final_graph = graph.compile(checkpointer=checkpointer)
final_graph

In [ ]:
from langchain_core.runnables import RunnableConfig

config: RunnableConfig = {
    "configurable": {
        "thread_id": "thread1"
    }
}

res1 = final_graph.invoke(
    {"messages": [HumanMessage(content="What is LangGraph used for?")]},
    config=config
)

res2 = final_graph.invoke(
    {"messages": [HumanMessage(content="Explain short-term memory in LangGraph.")]},
    config=config
)

res3 = final_graph.invoke(
    {"messages": [HumanMessage(content="How does message trimming work?")]},
    config=config
)

res4 = final_graph.invoke(
    {"messages": [HumanMessage(content="What is the purpose of a checkpointer?")]},
    config=config
)

res5 = final_graph.invoke(
    {"messages": [HumanMessage(content="How can I reduce token usage in an LLM application?")]},
    config=config
)

res6 = final_graph.invoke(
    {"messages": [HumanMessage(content="Summarize the main concepts of this notebook.")]},
    config=config
)